# Extraction Asawan — version corrigée pour Google Colab

Cette version remplace l'ancien notebook. Si tu vois encore une URL sans guillemets dans le code, tu n'es pas dans cette version.

Le fichier EXE est téléchargé comme donnée et n'est jamais exécuté.

Source officielle : https://www.asawan.org/fr/lexique_soninke


In [ ]:
from pathlib import Path
import requests, subprocess, os, json, re, shutil, textwrap
WORK = Path('/content/asawan_lexique'); WORK.mkdir(exist_ok=True)
EXE = WORK / 'Lexique-Soninke-Francais-English.exe'
urls = [
    'https://www.asawan.org/sites/www.asawan.org/files/uploads/Lexique%20Soninke-Francais-English%2006-2016.exe',
    'https://asawan.org/Soninke-Lexique_Pro_Setup.exe',
]
download_error = None
for url in urls:
    try:
        resp = requests.get(url, timeout=90, headers={'User-Agent': 'Mozilla/5.0'}, allow_redirects=True)
        print('URL:', url, '| HTTP:', resp.status_code, '| type:', resp.headers.get('Content-Type'), '| bytes:', len(resp.content))
        resp.raise_for_status()
        # Reject HTML error/redirect pages saved under an .exe name.
        prefix = resp.content[:512].lstrip().lower()
        if prefix.startswith((b'<!doctype html', b'<html', b'<?xml')) and 'executable' not in resp.headers.get('Content-Type', '').lower():
            print('La réponse ressemble à une page web, pas au programme. On essaie la source suivante.')
            continue
        if len(resp.content) < 100_000:
            print('Fichier trop petit pour être le lexique attendu; on essaie la source suivante.')
            continue
        EXE.write_bytes(resp.content)
        print('Téléchargement réussi :', EXE, '|', EXE.stat().st_size, 'octets')
        break
    except Exception as e:
        download_error = repr(e)
        print('Échec pour cette adresse :', download_error)
if not EXE.exists():
    print('Téléchargement automatique non abouti. Tu pourras importer le fichier officiel dans la cellule suivante.')


In [ ]:
# Si le téléchargement a échoué, cette cellule permet de téléverser le .exe officiel.
if not EXE.exists():
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        p = Path(name)
        if p.suffix.lower() == '.exe' and p.stat().st_size > 100_000:
            shutil.copy2(p, EXE); print('Fichier importé:', EXE, EXE.stat().st_size, 'octets'); break
if not EXE.exists(): raise RuntimeError('Fichier absent. Télécharge le fichier officiel Asawan puis importe-le ici.')
print('Signature:', EXE.open('rb').read(16), '| taille:', EXE.stat().st_size)

In [ ]:
# Installer uniquement des outils d'inspection/extraction d'archives; aucun lancement du .exe.
import subprocess
subprocess.run('apt-get -qq update && apt-get -qq install -y p7zip-full innoextract cabextract unshield file', shell=True, check=True)
subprocess.run(['file', str(EXE)], check=False)
subprocess.run(['7z', 'l', str(EXE)], check=False)

In [ ]:
# Tentatives d'extraction inertes. Chaque outil lit le fichier comme données; il n'est pas exécuté.
OUT = WORK / 'extracted'; OUT.mkdir(exist_ok=True)
attempts = [
    ['7z','x','-y',f'-o{OUT}',str(EXE)],
    ['innoextract','-d',str(OUT),str(EXE)],
    ['cabextract','-q','-d',str(OUT),str(EXE)],
]
for cmd in attempts:
    print('\n>>>', ' '.join(cmd))
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        print('exit=',p.returncode, '\nstdout:',p.stdout[-2500:], '\nstderr:',p.stderr[-1200:])
    except Exception as e: print(type(e).__name__, e)
print('\nFichiers extraits:')
for p in sorted(OUT.rglob('*')):
    if p.is_file(): print(p.relative_to(OUT), p.stat().st_size, 'bytes')

In [ ]:
# Inventaire des formats susceptibles de contenir les entrées lexicales.
interesting = {'.db','.sqlite','.sqlite3','.mdb','.accdb','.csv','.tsv','.txt','.xml','.json','.html','.htm','.dat','.lex','.dic','.idx','.mdx','.mdd','.rtf','.doc','.docx','.xls','.xlsx'}
hits=[]
for p in OUT.rglob('*'):
    if p.is_file() and (p.suffix.lower() in interesting or any(k in p.name.lower() for k in ['lexicon','lexique','soninke','dictionary','dict'])):
        hits.append((str(p),p.stat().st_size,p.suffix.lower()))
for h in hits: print(h)
print('\nNombre de fichiers candidats:',len(hits))
print('Si une base de données ou des fichiers texte apparaissent, ne les fusionne pas encore: envoie-moi la liste affichée ou le fichier extrait pour que je construise un parseur contrôlé.')